# AIODOO — All Adapters Colab Pipeline

One notebook to train → validate/certify → (optionally package) every capability adapter.

**This notebook only orchestrates.** Algorithms live in:
- `aiodoo-training` — LoRA / QLoRA training
- `aiodoo-validation` — certification
- `aiodoo-model` — registry packaging (optional)

## Drive layout (required)

```text
My Drive/colab_notebooks/AIODOO/
  datasets/v1.0.0/          ← SFT JSONL (coding_v1_0.jsonl, …)
  experiments/
  logs/
  models/{adapters,merged,exports,registry,registry_storage,base}/
  training/                 ← aiodoo-training clone + cache/
```

## Current section

1. Environment + paths  
2. Mount Drive + install deps  
3. Workspace + clone training  
4. **Coding** train → validate/certify  
5. Template cell for the next adapter (planner, …)

Do **not** train on `evaluation_benchmark_catalog.jsonl`.

## 0) Master configuration — edit once

All paths and environment variables for this Colab runtime.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# =============================================================================
# MASTER SWITCHES
# =============================================================================
TRAINING_ID = "coding"  # change per section: coding → planner → … → evaluation
AUTO_RESUME = True      # True = resume from latest Drive checkpoint if present
RUN_VALIDATION = True
RUN_PACKAGING = False   # set True after aiodoo-model is installed
VALIDATION_TIER = "standard"  # smoke | standard | strict (per aiodoo-validation)

# Git refs (aiodoo-training defaults to frozen tag; override if you need local main)
AIODOO_TRAINING_REF = os.environ.get("AIODOO_TRAINING_REF", "v2.0.0")
AIODOO_TRAINING_URL = os.environ.get(
    "AIODOO_TRAINING_URL",
    "https://github.com/support-kitech/aiodoo-training.git",
)
AIODOO_COLAB_URL = os.environ.get(
    "AIODOO_COLAB_URL",
    "https://github.com/support-kitech/aiodoo-colab.git",
)
AIODOO_VALIDATION_URL = os.environ.get(
    "AIODOO_VALIDATION_URL",
    "https://github.com/support-kitech/aiodoo-validation.git",
)
AIODOO_MODEL_URL = os.environ.get(
    "AIODOO_MODEL_URL",
    "https://github.com/support-kitech/aiodoo-model.git",
)

# =============================================================================
# GOOGLE DRIVE PATHS (matches your layout)
# =============================================================================
# Mount point Colab creates:
DRIVE_MOUNT = Path("/content/drive")
# Your screenshot: My Drive / colab_notebooks / AIODOO
DRIVE_NOTEBOOKS = DRIVE_MOUNT / "MyDrive" / "colab_notebooks"
AIODOO_ROOT = DRIVE_NOTEBOOKS / "AIODOO"

DATASETS_ROOT = AIODOO_ROOT / "datasets"
DATASET_VERSION = "v1.0.0"
DATASET_PATH = DATASETS_ROOT / DATASET_VERSION  # AIODOO_COLAB_DATASET_PATH

MODELS_ROOT = AIODOO_ROOT / "models"
ADAPTERS_ROOT = MODELS_ROOT / "adapters"
MERGED_ROOT = MODELS_ROOT / "merged"
EXPORTS_ROOT = MODELS_ROOT / "exports"
REGISTRY_ROOT = MODELS_ROOT / "registry"
REGISTRY_STORAGE_ROOT = MODELS_ROOT / "registry_storage"

EXPERIMENTS_ROOT = AIODOO_ROOT / "experiments"
LOGS_ROOT = AIODOO_ROOT / "logs"
TRAINING_ROOT = AIODOO_ROOT / "training"
TRAINING_REPO = TRAINING_ROOT / "aiodoo-training"
TRAINING_CACHE = TRAINING_ROOT / "cache"

# Hugging Face base models on Colab local SSD (NOT Drive)
MODEL_CACHE_ROOT = Path("/content/aiodoo-model-cache")

# Runtime checkouts (local SSD — fast clone / pip -e)
RUNTIME_REPOS = Path("/content/aiodoo-repos")
COLAB_REPO = RUNTIME_REPOS / "aiodoo-colab"
VALIDATION_REPO = RUNTIME_REPOS / "aiodoo-validation"
MODEL_REPO = RUNTIME_REPOS / "aiodoo-model"

# Dataset file expected for current TRAINING_ID
DATASET_FILES = {
    "coding": "coding_v1_0.jsonl",
    "planner": "planner_v1_0.jsonl",
    "execution": "execution_dataset.jsonl",
    "repair": "repair_v1_0.jsonl",
    "context": "context_v1_0.jsonl",
    "conversation": "conversation_dataset.jsonl",
    "approval": "approval_dataset.jsonl",
    "evaluation": "evaluation_dataset.jsonl",  # NEVER evaluation_benchmark_catalog.jsonl
}

# =============================================================================
# ENVIRONMENT VARIABLES (exported for all child processes)
# =============================================================================
os.environ["AIODOO_WORKSPACE_ROOT"] = str(AIODOO_ROOT)
os.environ["AIODOO_COLAB_ROOT"] = str(COLAB_REPO)
os.environ["AIODOO_COLAB_DATASET_PATH"] = str(DATASET_PATH)
os.environ["AIODOO_COLAB_MODEL_CACHE"] = str(MODEL_CACHE_ROOT)
# AIODOO_COLAB_MODEL_PATH is set after ModelStore.ensure()
os.environ["HF_HOME"] = str(MODEL_CACHE_ROOT / "hf")
os.environ["TRANSFORMERS_CACHE"] = str(MODEL_CACHE_ROOT / "transformers")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(MODEL_CACHE_ROOT / "hub")
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Optional secrets (set in Colab Secrets or leave empty)
# from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print("TRAINING_ID           =", TRAINING_ID)
print("AIODOO_ROOT           =", AIODOO_ROOT)
print("DATASET_PATH          =", DATASET_PATH)
print("AIODOO_WORKSPACE_ROOT =", os.environ["AIODOO_WORKSPACE_ROOT"])
print("AIODOO_COLAB_DATASET_PATH =", os.environ["AIODOO_COLAB_DATASET_PATH"])
print("MODEL_CACHE_ROOT      =", MODEL_CACHE_ROOT)
print("TRAINING_REPO         =", TRAINING_REPO)
print("Expected dataset file =", DATASET_PATH / DATASET_FILES[TRAINING_ID])

## 1) Mount Google Drive + install runtime dependencies

In [ ]:
from google.colab import drive

drive.mount(str(DRIVE_MOUNT), force_remount=False)

assert DRIVE_NOTEBOOKS.is_dir(), f"Missing Drive notebooks folder: {DRIVE_NOTEBOOKS}"
assert AIODOO_ROOT.is_dir(), (
    f"Missing AIODOO workspace: {AIODOO_ROOT}\n"
    "Create MyDrive/colab_notebooks/AIODOO with datasets/, models/, …"
)
assert DATASET_PATH.is_dir(), f"Missing dataset version root: {DATASET_PATH}"

print("Drive mounted. Workspace OK:", AIODOO_ROOT)

In [ ]:
import subprocess

RUNTIME_REPOS.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)


def _git_clone_or_update(url: str, dest: Path, ref: str | None = None) -> None:
    if (dest / ".git").is_dir():
        subprocess.run(["git", "-C", str(dest), "fetch", "--all", "--tags"], check=True)
        if ref:
            subprocess.run(["git", "-C", str(dest), "checkout", ref], check=True)
            subprocess.run(
                ["git", "-C", str(dest), "pull", "--ff-only"],
                check=False,
            )
    else:
        cmd = ["git", "clone", "--depth", "1"]
        if ref:
            cmd += ["--branch", ref]
        cmd += [url, str(dest)]
        subprocess.run(cmd, check=True)


# Orchestration + validation + model packaging repos on local SSD
_git_clone_or_update(AIODOO_COLAB_URL, COLAB_REPO)
_git_clone_or_update(AIODOO_VALIDATION_URL, VALIDATION_REPO)
_git_clone_or_update(AIODOO_MODEL_URL, MODEL_REPO)

# Training deps commonly needed for QLoRA on Colab (aiodoo-training is not pip-installed)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers",
        "accelerate",
        "peft",
        "bitsandbytes",
        "datasets",
        "trl",
        "safetensors",
        "sentencepiece",
        "pyyaml",
    ],
    check=True,
)

# aiodoo-validation is application-layout (sys.path). aiodoo-model is installable.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(MODEL_REPO)],
    check=True,
)

COLAB_PYTHON = COLAB_REPO / "python"
for p in (str(COLAB_PYTHON), str(VALIDATION_REPO)):
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ["AIODOO_COLAB_ROOT"] = str(COLAB_REPO)
print("sys.path ready. COLAB_REPO =", COLAB_REPO)

## 2) Prepare workspace + clone aiodoo-training onto Drive

In [ ]:
from artifacts import browse_training_artifacts, summarize_artifacts
from colab_logging import configure_logging, get_logger
from config import load_config
from experiments import ExperimentStore
from models import ModelStore
from packaging import ModelRegistry, publish_adapter, summarize_packaging
from repository import TrainingRepository
from trainer import build_training_context, run_training, summarize_result
from training_ui import TrainingMonitor
from validation import run_validation, summarize_validation
from workspace import prepare_workspace

configure_logging()
logger = get_logger()

# drive_mount_root = …/MyDrive/colab_notebooks  → aiodoo_root = …/AIODOO
config = load_config(
    drive_mount_root=DRIVE_NOTEBOOKS,
    training_repository_url=AIODOO_TRAINING_URL,
    model_cache_root=MODEL_CACHE_ROOT,
    default_branch=AIODOO_TRAINING_REF,
    auto_mount_drive=False,  # already mounted above
)

workspace = prepare_workspace(config, mount=False)
assert workspace.root == AIODOO_ROOT, (workspace.root, AIODOO_ROOT)

print("workspace.root        =", workspace.root)
print("workspace.datasets    =", workspace.datasets)
print("workspace.models      =", workspace.models)
print("workspace.experiments =", workspace.experiments)
print("workspace.training    =", workspace.training)
print("workspace.model_cache =", workspace.model_cache)
print("AIODOO_WORKSPACE_ROOT =", os.environ["AIODOO_WORKSPACE_ROOT"])
print("AIODOO_COLAB_DATASET_PATH =", os.environ["AIODOO_COLAB_DATASET_PATH"])

In [ ]:
repo = TrainingRepository.from_workspace(workspace, config)
if repo.exists():
    repo.update()
else:
    repo.clone()
repo.verify()

print("aiodoo-training at:", repo.path)
print("git ref configured:", AIODOO_TRAINING_REF)
print("train.py exists:", (repo.path / "train.py").is_file())

In [ ]:
# Verify SFT file for current TRAINING_ID is on Drive
dataset_file = DATASET_PATH / DATASET_FILES[TRAINING_ID]
assert dataset_file.is_file(), (
    f"Missing dataset file: {dataset_file}\n"
    f"Upload {DATASET_FILES[TRAINING_ID]} under {DATASET_PATH}"
)
size_mb = dataset_file.stat().st_size / (1024 * 1024)
print(f"OK {dataset_file.name} ({size_mb:.1f} MB)")

# Hard guard: never point training at the benchmark catalog
catalog = DATASET_PATH / "evaluation_benchmark_catalog.jsonl"
print("Benchmark catalog present (do not train):", catalog.is_file())

---
# SECTION A — Coding adapter

`TRAINING_ID = "coding"`  
Dataset: `datasets/v1.0.0/coding_v1_0.jsonl`  
Published adapter: `models/adapters/aiodoo-coding/`

In [ ]:
TRAINING_ID = "coding"
os.environ["AIODOO_COLAB_DATASET_PATH"] = str(DATASET_PATH)

experiments = ExperimentStore(workspace=workspace)
experiment = experiments.load(TRAINING_ID)

print("training_id     =", experiment.training_id)
print("model_id        =", experiment.model_id)
print("dataset_version =", experiment.dataset_version)
print("config path hint:", workspace.training_repository / "configs" / "training" / TRAINING_ID)

In [ ]:
assert isinstance(experiment.model_id, str) and experiment.model_id.strip()
model_store = ModelStore(workspace=workspace, model_id=experiment.model_id)
model_path = model_store.ensure()

os.environ["AIODOO_COLAB_MODEL_PATH"] = str(model_path)
print("AIODOO_COLAB_MODEL_PATH =", os.environ["AIODOO_COLAB_MODEL_PATH"])
print("Base model ready at:", model_path)

### A.1 Train coding (aiodoo-training subprocess)

Child process env includes:
- `AIODOO_WORKSPACE_ROOT`
- `AIODOO_COLAB_DATASET_PATH`
- `AIODOO_COLAB_MODEL_PATH`
- `PYTHONUNBUFFERED=1`

In [ ]:
context = build_training_context(workspace, experiment, model_path=model_path)

print("dataset_path   =", context.dataset_path)
print("adapter_output =", context.adapter_output)
print("checkpoints    =", context.checkpoints_output)
print("logs_output    =", context.logs_output)
print("config         =", context.training_config_path)

monitor = TrainingMonitor(
    training_id=experiment.training_id,
    model_name=experiment.model_id,
    dataset_version=str(experiment.dataset_version),
)
monitor.display()
result = run_training(context, auto_resume=AUTO_RESUME, on_log_line=monitor.on_line)
monitor.finish(result)
print(summarize_result(result))

assert result.success, f"Training failed: {result.message}"
print("CODING TRAIN OK →", result.adapter_path)

In [ ]:
run_artifacts = browse_training_artifacts(workspace, experiment.training_id)
print(summarize_artifacts(run_artifacts))

### A.2 Validate / certify coding (aiodoo-validation)

Uses the adapter + base model paths produced by the training cell.

In [ ]:
if not RUN_VALIDATION:
    print("RUN_VALIDATION=False — skipped")
else:
    validation_outcome = run_validation(
        context,
        result,
        execution_tier=VALIDATION_TIER,
    )
    print(summarize_validation(validation_outcome))
    print("successful =", validation_outcome.successful)
    print("certified  =", validation_outcome.certified)
    if not validation_outcome.successful:
        raise RuntimeError("Coding validation/certification did not succeed")

### A.3 Optional — publish coding adapter into aiodoo-model registry

In [ ]:
if not RUN_PACKAGING:
    print("RUN_PACKAGING=False — skipped (set True when ready)")
else:
    registry = ModelRegistry.from_workspace(workspace)
    packaging_result = publish_adapter(registry, experiment, result)
    print(summarize_packaging(packaging_result))

---
# SECTION B — Next adapter template (copy/run for each capability)

Recommended order (smallest → largest):

1. `repair` — train only; Colab cert builder N/A  
2. `execution`  
3. `planner`  
4. `approval`  
5. `conversation`  
6. `context` — train only; no validation profile (use `AUTO_RESUME=True`)  
7. `evaluation` — longest; judgment SFT only (`evaluation_dataset.jsonl`); never the catalog  

For evaluation after Phases 1–4, set `AIODOO_TRAINING_REF` to a ref that includes the contract migration (not frozen `v2.0.0` if that tag predates it).
**How to use:** set `TRAINING_ID` below, run the cell. Reuse the same env vars from Section 0.

In [ ]:
# ---------------------------------------------------------------------------
# NEXT ADAPTER — change TRAINING_ID and run this cell after coding succeeds
# ---------------------------------------------------------------------------
TRAINING_ID = "planner"  # repair | execution | planner | approval | conversation | context | evaluation

os.environ["AIODOO_WORKSPACE_ROOT"] = str(AIODOO_ROOT)
os.environ["AIODOO_COLAB_DATASET_PATH"] = str(DATASET_PATH)
os.environ["PYTHONUNBUFFERED"] = "1"

dataset_file = DATASET_PATH / DATASET_FILES[TRAINING_ID]
assert dataset_file.is_file(), f"Missing {dataset_file}"
if TRAINING_ID == "evaluation":
    assert dataset_file.name == "evaluation_dataset.jsonl"

experiments = ExperimentStore(workspace=workspace)
experiment = experiments.load(TRAINING_ID)
model_store = ModelStore(workspace=workspace, model_id=experiment.model_id)
model_path = model_store.ensure()
os.environ["AIODOO_COLAB_MODEL_PATH"] = str(model_path)

context = build_training_context(workspace, experiment, model_path=model_path)
monitor = TrainingMonitor(
    training_id=experiment.training_id,
    model_name=experiment.model_id,
    dataset_version=str(experiment.dataset_version),
)
monitor.display()
result = run_training(context, auto_resume=AUTO_RESUME, on_log_line=monitor.on_line)
monitor.finish(result)
print(summarize_result(result))
assert result.success, result.message

print(summarize_artifacts(browse_training_artifacts(workspace, experiment.training_id)))

if RUN_VALIDATION:
    # Colab builders: coding/planner/approval/conversation/evaluation/execution.
    # No builder for repair; context is not a validation profile.
    _SKIP_CERT = {"context", "repair"}
    if TRAINING_ID in _SKIP_CERT:
        print(f"{TRAINING_ID}: training OK; Colab certification builder N/A — skipped")
    else:
        outcome = run_validation(context, result, execution_tier=VALIDATION_TIER)
        print(summarize_validation(outcome))
        if not outcome.successful:
            raise RuntimeError(f"{TRAINING_ID} validation failed")

if RUN_PACKAGING:
    registry = ModelRegistry.from_workspace(workspace)
    print(summarize_packaging(publish_adapter(registry, experiment, result)))

print(f"DONE: {TRAINING_ID} → {result.adapter_path}")

## Env var quick reference

| Variable | Value in this notebook |
|----------|------------------------|
| `AIODOO_WORKSPACE_ROOT` | `/content/drive/MyDrive/colab_notebooks/AIODOO` |
| `AIODOO_COLAB_DATASET_PATH` | `…/AIODOO/datasets/v1.0.0` |
| `AIODOO_COLAB_MODEL_PATH` | local HF snapshot under `/content/aiodoo-model-cache/…` |
| `AIODOO_COLAB_ROOT` | `/content/aiodoo-repos/aiodoo-colab` |
| `HF_HOME` / `TRANSFORMERS_CACHE` / `HUGGINGFACE_HUB_CACHE` | under model cache |
| `PYTHONUNBUFFERED` | `1` |

Training checkpoints: `AIODOO/training/cache/<id>/checkpoints/`  
Published adapters: `AIODOO/models/adapters/aiodoo-<id>/`